Download all the requirements

In [ ]:
%pip install torch==2.7.1 torchvision torchaudio==2.7.1+cu126 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
%pip install torchcodec ffmpeg-python pandas tqdm scikit-learn librosa birdnetlib birdnet resampy

In [ ]:
%pip freeze > requirements.txt

In [ ]:
# Download dataset
import kagglehub

# Download latest version
kagglehub.auth.set_kaggle_api_token('KGAT_fb4d6921c565524358c1914efc082ed4')
path = kagglehub.competition_download('birdclef-2026', output_dir=os.path.join("birdclef-2026"))

print("Path to competition files:", path)

Dataset and CNN

In [ ]:
# DataSet & DataLoader
import os
import ast
import torch
import torchaudio
from torch.utils.data import Dataset

class BirbSet(Dataset):
    # Not gonna pass the sample rate. I trust that the data is formatted at 32k as the competition says.
    def __init__(self, df, root, clip_length, label_to_idx, is_train=False):
        # Needed for the sake of the dataset itself
        self.clips            = []
        self.start_times      = []
        self.end_times        = []
        # Info from the csv file
        self.labels           = []
        self.secondary_labels = []
        self.ratings          = [] # Consider using this field somehow

        self.clip_length      = clip_length   # How long we want each chunk to be. Default to 5 seconds for competition standard
        self.sample_rate      = 32000         # Carried from the competition data description
        self.label_to_idx     = label_to_idx
        self.is_train         = is_train
        self.root             = root

        # First Augmentation: SpecAugment! Uncomment later
        # Spectrogram transforms
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.mel_spect = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=800,
            n_mels=64
        )

        # # Augmentation transforms
        # self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=40)
        # self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=16)
        
        # Build manifest arrays
        for _, entry in df.iterrows():

            # The root has to be the train_audio folder in this case
            curr_audio_loc = os.path.join(self.root, os.path.normpath(entry["filename"]))
            # We need to separate each file into blocks of chunk_size
            info = torchaudio.info(curr_audio_loc)
            duration = info.num_frames / self.sample_rate # Trust that sample rate is 32k
            
            pos = 0.0
            # Keep going until we have processed the entire duration
            while pos < duration:
                # If the remaining audio is less than clip_length, cap it at duration
                end_pos = min(pos + self.clip_length, duration)
                
                self.clips.append(curr_audio_loc)
                self.labels.append(self.label_to_idx[entry['primary_label']])
                self.ratings.append(entry.get('rating'))
                self.secondary_labels.append(entry.get('secondary_labels', '[]'))
                
                self.start_times.append(pos)                     
                self.end_times.append(end_pos)     
                
                # Advance by clip_length to check the next segment
                pos += self.clip_length
            

    def __len__(self):
        return len(self.clips)
    
    def __getitem__(self, idx):
        audio_clip = self.clips[idx]
        try:
            frame_offset = int(self.start_times[idx] * self.sample_rate)
            num_frames   = int((self.end_times[idx] - self.start_times[idx]) * self.sample_rate)

            waveform, _ = torchaudio.load(
                audio_clip, frame_offset=frame_offset, num_frames=num_frames
            )

            # Pad or truncate waveform to exact chunk size
            chunk_size  = int(self.sample_rate * self.clip_length) # Wrap in int because the clip_length may be something like 3.2 if it's at the end of the file for example
            current_len = waveform.shape[1]

            if current_len > chunk_size:
                waveform = waveform[:, :chunk_size]
            elif current_len < chunk_size:
                waveform = torch.nn.functional.pad(waveform, (0, chunk_size - current_len))

            # Create Spectrogram
            spectrogram = self.mel_spect(waveform)
            spectrogram = self.amp_to_db(spectrogram)
            
            # Standardize Spectrogram
            mean, std   = spectrogram.mean(), spectrogram.std() + 1e-6
            spectrogram = (spectrogram - mean) / std

            # Initialize target vector
            target = torch.zeros(len(self.label_to_idx), dtype=torch.float32)

            # Set primary label
            primary = self.labels[idx]
            target[primary] = 1.0 if self.ratings[idx] == 0 else self.ratings[idx] * 0.25

            # Set secondary labels with smoothed targets (e.g., 0.3)
            raw_secondary = self.secondary_labels[idx]
            if raw_secondary and raw_secondary not in ('[]', '', None):
                for sec_label in ast.literal_eval(raw_secondary):   
                    if sec_label in self.label_to_idx:
                        # FIX: Soft-label background birds to prevent over-confidence
                        target[self.label_to_idx[sec_label]] = 0.25 if self.ratings[idx] == 0 else self.ratings[idx] * 0.05

            # # FIX: Ensure ALL augmentations are strictly within the is_train block
            # if self.is_train:
            #     spectrogram = self.freq_mask(spectrogram)
            #     spectrogram = self.time_mask(spectrogram)

            #     # Gaussian noise 
            #     if torch.rand(1).item() < 0.5:
            #         spectrogram = spectrogram + torch.randn_like(spectrogram) * 0.1

            #     # Random gain 
            #     gain = torch.empty(1).uniform_(0.75, 1.25)
            #     spectrogram = spectrogram * gain

            return spectrogram, target
            
        except Exception as e:
            print(f"Skipping corrupted/missing file at index {idx} -> {e}. Path: {audio_clip}")
            # FIX: Prevent DataLoader crash by returning an adjacent random index
            return self.__getitem__((idx + 1) % len(self))

In [ ]:
# CNN
import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

class EfficientBirbNN(nn.Module):
    # It is 234 according to the competition description
    def __init__(self, num_classes = 234, pretrained=True):
        super().__init__()
        
        # 1. Load the base EfficientNet model
        weights = EfficientNet_B3_Weights.DEFAULT if pretrained else None
        self.base_model = efficientnet_b3(weights=weights)
        
        # 2. Modify the first convolutional layer to accept 1-channel spectrograms
        # EfficientNet's first layer is located at self.base_model.features[0][0]
        original_conv = self.base_model.features[0][0]
        self.base_model.features[0][0] = nn.Conv2d(
            in_channels=1, 
            out_channels=original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=False
        )
                
        # 3. Modify the final classification layer for your specific number of bird classes
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.4), # Extra dropout to prevent overfitting on audio data
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

In [ ]:
import os
import pandas as pd
from torch.utils.data import DataLoader
# Requires scikit-learn: pip install scikit-learn
from sklearn.model_selection import StratifiedGroupKFold 

# --- Configuration ---
root_path       = os.path.join("..", "birdclef-2026")
CLIP_LENGTH_SEC = 5.0  # Updated to 5.0s to match BirdCLEF 2026 evaluation windows

# 1. Load the master CSV
full_df = pd.read_csv(os.path.join(root_path, "train.csv"))

# 2. Universal label mapping (sorted for reproducibility)
# We can get this from taxonomy.csv
unique_labels = pd.read_csv(os.path.join(root_path, "taxonomy.csv"))
master_label_to_idx = {label: i for i, label in enumerate(unique_labels['primary_label'])}
num_classes         = len(master_label_to_idx)

# 3. Stratified Group Split (Anti-Leakage Validation Scheme)
# This guarantees every class appears in both halves while keeping 
# multiple chunks from the same audio file completely isolated together.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# We use 'filename' as the grouping key so unique files aren't split across train/val
train_indices, val_indices = next(
    sgkf.split(X=full_df, y=full_df['primary_label'], groups=full_df['filename'])
)

df_train = full_df.iloc[train_indices].reset_index(drop=True)
df_val   = full_df.iloc[val_indices].reset_index(drop=True)

print(f"Train samples: {len(df_train)} | Validation samples: {len(df_val)}")

# 5. DataLoaders
dset_train = BirbSet(
    df=df_train, 
    root=os.path.join(root_path, 'train_audio'), 
    clip_length=CLIP_LENGTH_SEC,
    label_to_idx=master_label_to_idx, 
    is_train=True  # Enables frequency/time masking & audio augmentations
)
loader = DataLoader(dset_train, batch_size=32, shuffle=True, pin_memory=True)

dset_val = BirbSet(
    df=df_val, 
    root=os.path.join(root_path, 'train_audio'), 
    clip_length=CLIP_LENGTH_SEC,
    label_to_idx=master_label_to_idx, 
    is_train=False  # Keeps validation data pristine and predictable
)
loader_val = DataLoader(dset_val, batch_size=32, shuffle=False, pin_memory=True)

Training and validation

In [ ]:
# Hyper params
import torch
device = torch.device("cuda")
model = EfficientBirbNN().to(device)
optimiser = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

In [ ]:
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler  # Updated API for modern PyTorch
from tqdm.notebook import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score

# Loss & Scaler setup
criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler('cuda')

def train_epoch(model, dataloader, optimizer, epoch):
    model.train()
    total_loss = 0.0
    
    # Check if dataloader is empty/hanging right away
    if len(dataloader) == 0:
        print("Warning: DataLoader has 0 batches. Check your dataset.")
        return 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", dynamic_ncols=True)
    
    for spectrograms, targets in pbar:
        # 1. Ensure shapes and types match exactly for BCEWithLogitsLoss
        spectrograms = spectrograms.to('cuda', non_blocking=True)
        targets = targets.to('cuda', dtype=torch.float32, non_blocking=True)
        
        # If targets are 1D (batch_size) and logits are 2D (batch_size, classes)
        if targets.ndim == 1:
            targets = targets.unsqueeze(1) 
        
        optimizer.zero_grad(set_to_none=True) # Slightly faster than zero_grad()
        
        # Mixed precision
        with autocast('cuda'):
            logits = model(spectrograms)
            loss = criterion(logits, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loss_val = loss.item()
        total_loss += loss_val
        pbar.set_postfix({'loss': f"{loss_val:.4f}"})
        
    return total_loss / len(dataloader)


@torch.no_grad()
def validate_epoch(model, dataloader, epoch):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]")
    
    for spectrograms, targets in pbar:
        spectrograms = spectrograms.to('cuda', non_blocking=True)
        targets = targets.to('cuda', dtype=torch.float32, non_blocking=True)
        
        if targets.ndim == 1:
            targets = targets.unsqueeze(1)
            
        with autocast('cuda'):
            logits = model(spectrograms)
            loss = criterion(logits, targets)
            
        total_loss += loss.item()
        
        # Apply sigmoid before pushing to CPU to keep calculations on GPU
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
        all_targets.append(targets.cpu().numpy())
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    # Concatenate all batches
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    
    # BirdCLEF metric safeguard:
    # Remove classes that have NO positive samples in this validation fold,
    # otherwise sklearn's macro ROC-AUC will crash or return skewed results.
    binary_targets = (all_targets > 0.5).astype(int)
    valid_classes = np.any(binary_targets == 1, axis=0) & np.any(binary_targets == 0, axis=0)
    
    if not np.any(valid_classes):
        print("Warning: No valid classes found for AUC calculation in this split.")
        val_auc = 0.5
    else:
        try:
            # Calculate macro AUC only on classes present in the validation split
            val_auc = roc_auc_score(
                binary_targets[:, valid_classes], 
                all_preds[:, valid_classes], 
                average='macro'
            )
        except ValueError as e:
            print(f"AUC calculation error: {e}")
            val_auc = 0.0
        
    return total_loss / len(dataloader), val_auc

In [ ]:
for i in range(15):
    train_epoch(model, loader, optimiser, i+1)
    validate_epoch(model, loader_val, i+1)